# ETL local — CSV quảng cáo → một CSV đã xử lý

Notebook thực hiện **Extract → Transform → Validate → Load**, có thể chạy từ đầu bằng **Run All**. Mỗi dòng vẫn là một quan sát nguồn; không gọi mã kỹ thuật là campaign_id và không gộp các dòng cùng năm chiều.

Chỉ có **một file dữ liệu đầu ra**, theo đường dẫn cấu hình trong thư mục con của `data/`. Bảng profiling, cảnh báo và đối chiếu được hiển thị trong notebook, không xuất thành CSV riêng. Không kết nối database hoặc sử dụng dịch vụ bên ngoài.

**Quy tắc đầu vào** được khai báo tập trung trong cell cấu hình bên dưới: ánh xạ cột, kiểu, công thức, dung sai và các tham số. Notebook ưu tiên `files.notebook_etl_config`, hoặc `files.etl_config` nếu YAML có đủ các mục của cấu hình notebook. YAML thuộc pipeline PostgreSQL cũ không thay thế cấu hình CSV này. Code không viết sẵn tên nhóm, năm, số dòng hoặc kết quả mong đợi. Tên cột vẫn cần khai báo một lần vì chương trình không thể tự đoán ý nghĩa nghiệp vụ.

**Giả định:** chi phí và doanh thu dùng chung đơn vị tiền để đối chiếu POC; CSV chưa xác nhận USD hoặc quy đổi. KPI theo dòng dùng cho kiểm tra; khi tổng hợp trong BI phải chia tổng tử cho tổng mẫu, không lấy trung bình KPI theo dòng.

## Cấu hình tập trung

Cell này chứa mapping cột, quy tắc tính và bảng ngưỡng từ phụ lục C. Đây là tham số nghiệp vụ để chỉnh ở một chỗ; các cell xử lý tra cấu hình, không gắn tên cột hoặc nhóm vào thuật toán. Bản notebook trên Git chạy được với dữ liệu nguồn và `config/project.yaml` đã có trong repo, không cần thêm hai file cấu hình mới.

In [1]:
import yaml

DEFAULT_CFG = yaml.safe_load("""
# Input contract: physical column names are declared once here.
# Values/counts/dates/categories are discovered from the actual CSV.
source:
  encoding: utf-8-sig
  date_format: '%Y-%m-%d'
  assume_common_currency: true
dimensions:
  date: {column: date, type: date}
  platform: {column: platform, type: category}
  campaign_type: {column: campaign_type, type: category}
  industry: {column: industry, type: category}
  country: {column: country, type: category}
measures:
  impressions: {column: impressions, type: integer}
  clicks: {column: clicks, type: integer}
  ad_spend: {column: ad_spend, type: decimal}
  conversions: {column: conversions, type: integer}
  revenue: {column: revenue, type: decimal}
ratios:
  ctr: {numerator: clicks, denominator: impressions, source: CTR, source_decimals: 4}
  cpc: {numerator: ad_spend, denominator: clicks, source: CPC, source_decimals: 2}
  conversion_rate: {numerator: conversions, denominator: clicks}
  cpa: {numerator: ad_spend, denominator: conversions, source: CPA, source_decimals: 2}
  roas: {numerator: revenue, denominator: ad_spend, source: ROAS, source_decimals: 2}
transform:
  trim_dimensions: true
  integer_bits: 64
  money_precision: 18
  money_scale: 2
  date_role: date
  spend_role: ad_spend
  iqr_multiplier: 1.5
  outlier_roles: [ad_spend, revenue, roas]
  ordered_counts:
    - {left: clicks, right: impressions}
    - {left: conversions, right: clicks}
  derived_columns:
    observation_id: observation_id
    source_record: source_record
    source_hash: source_sha256
    year: year
    quarter: quarter
    month: year_month
    month_covered: flag_month_not_covered
    dimension_repeat: flag_repeated_dimensions
    full_duplicate: flag_identical_source_row
    normalized: flag_dimension_normalized
  ratio_suffix: _calculated
  flag_prefix: flag_
spend_levels:
  policy_file: config/spend_thresholds.yaml
  dimensions: [country, industry]
  low_bound: low_max
  high_bound: medium_max
  labels: [Low, Medium, High]
  unknown_label: Unclassified
  column: spend_level
display:
  preview_rows: 5
export:
  directory: data/processed
  filename: global_ads_performance_clean.csv
  encoding: utf-8-sig
""")

DEFAULT_POLICY = yaml.safe_load("""
provenance: Appendix C supplied by the user; thresholds are policy inputs, not inferred
  from this CSV.
records:
- {country: UAE, industry: Fintech, low_max: 5239.86, medium_max: 13582.33}
- {country: UAE, industry: EdTech, low_max: 3823.26, medium_max: 10231.12}
- {country: UAE, industry: Healthcare, low_max: 5706.85, medium_max: 14074.34}
- {country: UAE, industry: SaaS, low_max: 4595.14, medium_max: 13020.79}
- {country: UAE, industry: E-commerce, low_max: 4200.44, medium_max: 8729.09}
- {country: UK, industry: Fintech, low_max: 5536.5, medium_max: 12606.41}
- {country: UK, industry: EdTech, low_max: 5063.22, medium_max: 13988.92}
- {country: UK, industry: Healthcare, low_max: 9643.62, medium_max: 27121.52}
- {country: UK, industry: SaaS, low_max: 4089.83, medium_max: 12674.55}
- {country: UK, industry: E-commerce, low_max: 8688.3, medium_max: 26228.45}
- {country: USA, industry: Fintech, low_max: 5907.65, medium_max: 13243.1}
- {country: USA, industry: EdTech, low_max: 5582.0, medium_max: 13663.0}
- {country: USA, industry: Healthcare, low_max: 5915.4, medium_max: 14207.73}
- {country: USA, industry: SaaS, low_max: 7301.5, medium_max: 16606.21}
- {country: USA, industry: E-commerce, low_max: 6309.51, medium_max: 19677.66}
- {country: Germany, industry: Fintech, low_max: 7062.12, medium_max: 14971.0}
- {country: Germany, industry: EdTech, low_max: 4748.0, medium_max: 11112.61}
- {country: Germany, industry: Healthcare, low_max: 6241.8, medium_max: 17664.98}
- {country: Germany, industry: SaaS, low_max: 5602.61, medium_max: 12297.3}
- {country: Germany, industry: E-commerce, low_max: 5845.37, medium_max: 14827.69}
- {country: Canada, industry: Fintech, low_max: 4824.85, medium_max: 12737.67}
- {country: Canada, industry: EdTech, low_max: 5616.13, medium_max: 12641.22}
- {country: Canada, industry: Healthcare, low_max: 6465.52, medium_max: 16244.68}
- {country: Canada, industry: SaaS, low_max: 6139.42, medium_max: 15288.2}
- {country: Canada, industry: E-commerce, low_max: 7987.52, medium_max: 19375.01}
- {country: India, industry: Fintech, low_max: 6115.37, medium_max: 14548.45}
- {country: India, industry: EdTech, low_max: 6458.93, medium_max: 14393.57}
- {country: India, industry: Healthcare, low_max: 7727.48, medium_max: 18075.97}
- {country: India, industry: SaaS, low_max: 4195.51, medium_max: 9207.93}
- {country: India, industry: E-commerce, low_max: 4174.14, medium_max: 9408.49}
- {country: Australia, industry: Fintech, low_max: 6436.28, medium_max: 15810.58}
- {country: Australia, industry: EdTech, low_max: 6970.83, medium_max: 14690.5}
- {country: Australia, industry: Healthcare, low_max: 8150.48, medium_max: 19493.45}
- {country: Australia, industry: SaaS, low_max: 5909.15, medium_max: 16478.76}
- {country: Australia, industry: E-commerce, low_max: 9347.29, medium_max: 21979.43}
""")

In [2]:
import csv
import hashlib
import io
import os
import platform
import tempfile
from decimal import Decimal, InvalidOperation, localcontext
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'config/project.yaml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Mở notebook từ thư mục dự án hoặc thư mục con.')
ROOT = ROOT.resolve()
with (ROOT / 'config/project.yaml').open(encoding='utf-8') as file:
    project = yaml.safe_load(file)
cfg = DEFAULT_CFG
config_file = project['files'].get('notebook_etl_config', project['files'].get('etl_config'))
if config_file:
    with (ROOT / config_file).open(encoding='utf-8') as file:
        candidate = yaml.safe_load(file)
    if isinstance(candidate, dict) and set(DEFAULT_CFG) <= set(candidate):
        cfg = candidate
fields = {**cfg['dimensions'], **cfg['measures']}
col = {key: spec['column'] for key, spec in fields.items()}
if len(fields) != len(cfg['dimensions']) + len(cfg['measures']) or len(set(col.values())) != len(col):
    raise ValueError('ID hoặc ánh xạ cột bị trùng.')
tx, derived = cfg['transform'], cfg['transform']['derived_columns']
date_col = col[tx['date_role']]
ratio_cols = {key: key + tx['ratio_suffix'] for key in cfg['ratios']}
if any(not {spec['numerator'], spec['denominator']} <= set(cfg['measures'])
       for spec in cfg['ratios'].values()):
    raise ValueError('Công thức tham chiếu measure không tồn tại.')
source_path = ROOT / project['paths']['raw_data_dir'] / project['files']['raw_dataset']
output_dir = (ROOT / cfg['export']['directory']).resolve()
output_path = output_dir / cfg['export']['filename']
if (ROOT / 'data').resolve() not in output_dir.parents or output_path.parent != output_dir:
    raise ValueError('Đầu ra phải là một file trong thư mục con của data/.')
if output_path.resolve() == source_path.resolve():
    raise ValueError('Không được ghi đè dữ liệu nguồn.')
pd.set_option('display.max_columns', 16)
display(Markdown(f'Python **{platform.python_version()}**, pandas **{pd.__version__}**. '
                 f'Đầu ra: `{output_path.relative_to(ROOT)}`.'))

<IPython.core.display.Markdown object>


## 1. Extract — đọc nguyên trạng và xem cấu trúc

Đọc bytes một lần để tính hash và parse CSV. Header phải duy nhất, mỗi bản ghi có đủ trường và các cột bắt buộc phải thỏa mapping. Giữ `raw` nguyên dạng chuỗi để đối chiếu; chưa thay chuỗi rỗng bằng 0 và chưa xóa dòng.

Hash nhận diện phiên bản file; số thứ tự bản ghi nhận diện vị trí trong file đó, không phải số dòng văn bản khi một ô CSV có ký tự xuống dòng. Cột ngoài hợp đồng vẫn được giữ trong đầu ra nhưng không tự trở thành measure.

In [3]:
source_bytes = source_path.read_bytes()
source_hash = hashlib.sha256(source_bytes).hexdigest()
records = list(csv.reader(io.StringIO(source_bytes.decode(cfg['source']['encoding']), newline=''), strict=True))
if len(records) < 2:
    raise ValueError('Nguồn phải có header và ít nhất một quan sát.')
header, values = records[0], records[1:]
if len(header) != len(set(header)) or any(not name.strip() for name in header):
    raise ValueError('Header rỗng hoặc bị trùng.')
if any(len(row) != len(header) for row in values) or set(col.values()) - set(header):
    raise ValueError('Sai cấu trúc CSV hoặc thiếu cột bắt buộc trong mapping.')
raw = pd.DataFrame(values, columns=header)
audit = pd.DataFrame({
    'column': header, 'distinct': raw.nunique().values,
    'blank': raw.apply(lambda s: s.str.strip().eq('').sum()).values,
    'outer_whitespace': raw.apply(lambda s: s.ne(s.str.strip()).sum()).values,
})
display(audit)
display(raw.head(cfg['display']['preview_rows']))
display(Markdown(f'Nguồn có **{len(raw):,} quan sát**, **{len(header)} cột**. '
                 f'SHA-256: `{source_hash}`.'))

           column  distinct  blank  outer_whitespace
0            date       360      0                 0
1        platform         3      0                 0
2   campaign_type         4      0                 0
3        industry         5      0                 0
4         country         7      0                 0
5     impressions      1790      0                 0
6          clicks      1616      0                 0
7             CTR       624      0                 0
8             CPC       327      0                 0
9        ad_spend      1800      0                 0
10    conversions       518      0                 0
11            CPA      1634      0                 0
12        revenue      1798      0                 0
13           ROAS       999      0                 0
         date    platform campaign_type    industry  country impressions  \
0  2024-01-21  Google Ads        Search     Fintech      UAE       59886   
1  2024-01-22  TikTok Ads        Search      EdTech  

## 2. Transform — chuyển kiểu, không làm mất quan sát

Chuyển ngày theo định dạng đã khai báo; cột đếm phải là số nguyên không âm và nằm trong miền kiểu đã cấu hình. Chi phí/doanh thu dùng **Decimal**, không chuyển qua float để tránh sai số tiền. Giá trị vượt precision/scale được báo lỗi, không âm thầm làm tròn.

Chỉ trim thuộc tính phân loại khi bật quy tắc cấu hình; không tự đổi tên nền tảng, quốc gia hoặc gộp nhãn gần giống. Nếu có dữ liệu bắt buộc không hợp lệ, dừng trước xuất và giữ nguyên CSV đầu ra cũ nếu có.

In [4]:
def number(value, kind='decimal', money=False):
    result = Decimal(str(value))
    if not result.is_finite() or result < 0:
        raise ValueError('Số phải hữu hạn và không âm.')
    if kind == 'integer':
        if result != result.to_integral_value() or result >= 2 ** (tx['integer_bits'] - 1):
            raise ValueError('Không phải số nguyên hợp lệ.')
        return int(result)
    if money:
        precision, scale = tx['money_precision'], tx['money_scale']
        with localcontext() as context:
            context.prec = max(precision + scale + 4, len(result.as_tuple().digits) + scale + 4)
            if result != result.quantize(Decimal(1).scaleb(-scale)) or result >= Decimal(10) ** (precision - scale):
                raise ValueError('Giá trị tiền vượt precision/scale đã khai báo.')
    return result

clean = raw.copy(deep=True)
normalized = pd.Series(False, index=raw.index)
errors = []
for role, spec in fields.items():
    name = col[role]
    if spec['type'] == 'date':
        clean[name] = pd.to_datetime(raw[name], format=cfg['source']['date_format'], errors='coerce')
        errors += [{'record': int(i + 1), 'field': name} for i in raw.index[clean[name].isna()]]
    elif spec['type'] == 'category':
        clean[name] = raw[name].str.strip() if tx['trim_dimensions'] else raw[name]
        normalized |= clean[name].ne(raw[name])
        errors += [{'record': int(i + 1), 'field': name} for i in raw.index[clean[name].eq('')]]
    else:
        parsed = []
        for i, value in raw[name].items():
            try:
                parsed.append(number(value, spec['type'], money=spec['type'] == 'decimal'))
            except (InvalidOperation, ValueError, OverflowError):
                parsed.append(None)
                errors.append({'record': int(i + 1), 'field': name})
        clean[name] = pd.Series(parsed, index=raw.index, dtype=object)
if errors:
    display(pd.DataFrame(errors).head(cfg['display']['preview_rows']))
    raise ValueError(f'Có {len(errors)} giá trị bắt buộc không hợp lệ; chưa xuất dữ liệu.')
for role, spec in cfg['measures'].items():
    if spec['type'] == 'integer':
        clean[col[role]] = clean[col[role]].astype('int64')
display(pd.DataFrame([{'role': role, 'source_column': col[role], 'type': spec['type']}
                      for role, spec in fields.items()]))

            role  source_column      type
0           date           date      date
1       platform       platform  category
2  campaign_type  campaign_type  category
3       industry       industry  category
4        country        country  category
5    impressions    impressions   integer
6         clicks         clicks   integer
7       ad_spend       ad_spend   decimal
8    conversions    conversions   integer
9        revenue        revenue   decimal


## 3. KPI — giữ nguồn, tính lại và ghi cờ đối chiếu

KPI nguồn giữ nguyên. Cột hậu tố `_calculated` được tính từ tử/mẫu trong cấu hình. Mẫu số bằng 0 tạo giá trị trống, không infinity. Conversion Rate được bổ sung từ Conversions/Clicks.

So sánh KPI nguồn bằng Decimal với dung sai nửa đơn vị làm tròn theo `source_decimals`; sai số đúng bằng ngưỡng vẫn được chấp nhận. Cột nguồn tùy chọn không có thì trạng thái so sánh để trống. Lệch KPI hoặc KPI nguồn lỗi được gắn cờ, không xóa quan sát có các đại lượng cơ sở hợp lệ.

In [5]:
def divide(numerator, denominator):
    return Decimal(str(numerator)) / Decimal(str(denominator)) if denominator else None

def flag_name(name):
    return tx['flag_prefix'] + name

flags = {}
ratio_audit = []
for role, spec in cfg['ratios'].items():
    calculated = [divide(a, b) for a, b in zip(clean[col[spec['numerator']]], clean[col[spec['denominator']]])]
    clean[ratio_cols[role]] = pd.Series(calculated, index=clean.index, dtype=object)
    flags[flag_name(role + '_undefined')] = pd.Series([v is None for v in calculated], index=clean.index)
    source = spec.get('source')
    if not source:
        continue
    invalid = pd.Series(False, index=clean.index, dtype='boolean')
    mismatch = pd.Series(pd.NA, index=clean.index, dtype='boolean')
    if source in raw:
        tolerance = Decimal('0.5').scaleb(-spec['source_decimals'])
        for i, expected in enumerate(calculated):
            try:
                observed = number(raw.at[i, source])
                if expected is not None:
                    mismatch.at[i] = abs(observed - expected) > tolerance
            except (InvalidOperation, ValueError):
                invalid.at[i] = True
    else:
        invalid[:] = pd.NA
    flags[flag_name(role + '_source_mismatch')] = mismatch
    flags[flag_name(role + '_source_invalid')] = invalid
    ratio_audit.append({'metric': role, 'source_column': source, 'source_present': source in raw,
                        'mismatch_count': int(mismatch.fillna(False).sum()),
                        'invalid_source_count': int(invalid.fillna(False).sum())})
display(pd.DataFrame(ratio_audit))

  metric source_column  source_present  mismatch_count  invalid_source_count
0    ctr           CTR            True              47                     0
1    cpc           CPC            True               0                     0
2    cpa           CPA            True               0                     0
3   roas          ROAS            True               0                     0


## 4. Grain, thời gian và cảnh báo phân phối

Giữ mọi quan sát hợp lệ, kể cả các dòng cùng năm chiều. Cờ lặp chiều đánh dấu **tất cả thành viên** của nhóm, khác với số dòng dư khi đếm trùng. Dòng giống hoàn toàn cũng chỉ được gắn cờ vì chưa có khóa nghiệp vụ.

Mã quan sát ghép hash nguồn với thứ tự bản ghi, giúp tái lập khi chạy lại cùng file. Bổ sung năm, quý, tháng và cờ phạm vi nguồn chưa bao trọn tháng. Ngày vắng được báo trong notebook; không thêm dòng có doanh thu/chi phí bằng 0.

IQR chỉ đánh dấu giá trị ngoài khoảng thống kê trên toàn bộ nguồn. Đây là dấu hiệu cần xem xét, không phải khẳng định dữ liệu sai; không xóa hoặc cắt ngọn giá trị.

In [6]:
dimension_cols = [col[key] for key in cfg['dimensions']]
generated = list(derived.values()) + list(ratio_cols.values()) + list(flags) + [cfg['spend_levels']['column']]
extra_flags = [flag_name(role + '_iqr') for role in tx['outlier_roles']]
extra_flags += [flag_name(rule['left'] + '_above_' + rule['right']) for rule in tx['ordered_counts']]
extra_flags += [flag_name('spend_policy_missing')]
if len(generated + extra_flags) != len(set(generated + extra_flags)) or set(generated + extra_flags) & set(raw.columns):
    raise ValueError('Tên cột sinh thêm bị trùng; sửa cấu hình trước khi xuất.')
clean[derived['source_record']] = range(1, len(clean) + 1)
clean[derived['source_hash']] = source_hash
clean[derived['observation_id']] = source_hash + ':' + clean[derived['source_record']].astype(str)
flags[derived['dimension_repeat']] = clean.duplicated(dimension_cols, keep=False)
flags[derived['full_duplicate']] = raw.duplicated(keep=False)
flags[derived['normalized']] = normalized
dates = clean[date_col]
clean[derived['year']] = dates.dt.year
clean[derived['quarter']] = dates.dt.to_period('Q').astype(str)
clean[derived['month']] = dates.dt.to_period('M').astype(str)
periods = dates.dt.to_period('M')
flags[derived['month_covered']] = (periods.dt.start_time < dates.min()) | (periods.dt.end_time.dt.normalize() > dates.max())
missing_days = pd.date_range(dates.min(), dates.max()).difference(dates.unique())
outlier_audit = []
for role in tx['outlier_roles']:
    name = col[role] if role in cfg['measures'] else ratio_cols[role]
    values = pd.to_numeric(clean[name], errors='coerce').astype(float)
    q1, q3 = values.quantile([0.25, 0.75])
    margin = tx['iqr_multiplier'] * (q3 - q1)
    flags[flag_name(role + '_iqr')] = values.lt(q1 - margin) | values.gt(q3 + margin)
    outlier_audit.append({'metric': role, 'lower_bound': q1 - margin, 'upper_bound': q3 + margin,
                          'flagged': int(flags[flag_name(role + '_iqr')].sum())})
for rule in tx['ordered_counts']:
    flags[flag_name(rule['left'] + '_above_' + rule['right'])] = clean[col[rule['left']]].gt(clean[col[rule['right']]])
display(pd.DataFrame(outlier_audit))
display(Markdown(f'Có **{clean.duplicated(dimension_cols).sum()} dòng dư theo tổ hợp chiều**, '
                 f'**{flags[derived["dimension_repeat"]].sum()} quan sát thuộc các nhóm lặp**. '
                 'Tất cả được giữ lại.'))
display(Markdown('Ngày không xuất hiện trong khoảng nguồn: ' +
                 (', '.join(day.strftime('%Y-%m-%d') for day in missing_days) or 'Không có')))

     metric   lower_bound   upper_bound  flagged
0  ad_spend  -7767.276250  18189.693750       90
1   revenue -40255.683750  86494.826250      127
2      roas     -6.896192     17.271668      123
<IPython.core.display.Markdown object>
<IPython.core.display.Markdown object>


## 5. Spend Level — áp dụng bảng chính sách đã cung cấp

Đọc bảng ngưỡng Country × Industry có ghi nguồn phụ lục C. Ưu tiên YAML chính sách nếu có; nếu không, dùng bảng trong cell cấu hình. Tên cột nguồn được tra qua mapping; không viết tên quốc gia/ngành trong code. Low ≤ ngưỡng thấp; Medium lớn hơn ngưỡng thấp và ≤ ngưỡng cao; High lớn hơn ngưỡng cao.

Nếu nhóm chưa có chính sách, giữ nhãn Unclassified và cờ cảnh báo; không tự suy ra ngưỡng hoặc bỏ quan sát. Bảng chính sách chỉ nằm trong bộ nhớ, không xuất CSV riêng. Spend Level bổ sung ngữ cảnh phân tích, không phải nhãn cơ hội tăng trưởng.

In [7]:
sl = cfg['spend_levels']
policy_path = ROOT / sl['policy_file']
policy = (yaml.safe_load(policy_path.read_text(encoding='utf-8'))
          if policy_path.is_file() else DEFAULT_POLICY)
lookup = {}
for record in policy['records']:
    key = tuple(str(record[role]).strip() if tx['trim_dimensions'] else str(record[role])
                for role in sl['dimensions'])
    low, high = number(record[sl['low_bound']]), number(record[sl['high_bound']])
    if key in lookup or low >= high:
        raise ValueError('Chính sách có khóa trùng hoặc thứ tự ngưỡng không hợp lệ.')
    lookup[key] = (low, high)
if len(sl['labels']) != 3 or len(set(sl['labels'] + [sl['unknown_label']])) != 4:
    raise ValueError('Cần ba mức chi tiêu và một nhãn chưa phân loại khác nhau.')
levels = []
for key, spend in zip(clean[[col[role] for role in sl['dimensions']]].itertuples(index=False, name=None),
                      clean[col[tx['spend_role']]]):
    bounds = lookup.get(key)
    level = (sl['unknown_label'] if bounds is None else sl['labels'][0] if spend <= bounds[0]
             else sl['labels'][1] if spend <= bounds[1] else sl['labels'][2])
    levels.append(level)
clean[sl['column']] = levels
flags[flag_name('spend_policy_missing')] = clean[sl['column']].eq(sl['unknown_label'])
for name, values in flags.items():
    clean[name] = values
display(clean[sl['column']].value_counts().rename_axis('spend_level').to_frame('observations'))

             observations
spend_level              
Low                  1107
Medium                499
High                  194


## 6. Validate — đối chiếu trước khi xuất

Kiểm tra bảo toàn dòng và tổng từng đại lượng, mã quan sát duy nhất, không thay đổi file nguồn và không sửa các cột KPI nguồn. Kỳ vọng lấy từ dữ liệu nguồn, không viết cố định số dòng, số nhóm hoặc tổng POC.

Các cảnh báo lặp chiều, lệch KPI, IQR hoặc tháng chưa được bao trọn không tự làm một quan sát bị loại. Bảng dưới cho biết số cờ và số trường hợp chưa kiểm tra được. Độ bao phủ thời gian không chứng minh nguồn đã ghi nhận đủ hoạt động kinh doanh.

In [8]:
checks = [{'check': 'row_count_preserved', 'passed': len(clean) == len(raw)},
          {'check': 'observation_id_unique', 'passed': clean[derived['observation_id']].is_unique},
          {'check': 'columns_unique', 'passed': clean.columns.is_unique},
          {'check': 'source_unchanged', 'passed': hashlib.sha256(source_path.read_bytes()).hexdigest() == source_hash}]
totals = []
for role, spec in cfg['measures'].items():
    name = col[role]
    before = sum((number(v, spec['type'], money=spec['type'] == 'decimal') for v in raw[name]), Decimal(0))
    after = sum((Decimal(str(v)) for v in clean[name]), Decimal(0))
    totals.append({'measure': role, 'source_total': before, 'output_total': after})
    checks.append({'check': role + '_total_preserved', 'passed': before == after})
for spec in cfg['ratios'].values():
    source = spec.get('source')
    if source in raw:
        checks.append({'check': source + '_source_preserved', 'passed': clean[source].equals(raw[source])})
validation = pd.DataFrame(checks)
display(pd.DataFrame(totals))
display(validation)
display(pd.DataFrame([{'flag': name, 'flagged': int(values.fillna(False).sum()),
                       'not_evaluated': int(values.isna().sum())} for name, values in flags.items()]))
if not validation.passed.all():
    raise AssertionError('Đối chiếu thất bại; chưa xuất file.')
display(clean.head(cfg['display']['preview_rows']))

       measure source_total output_total
0  impressions    185254234    185254234
1       clicks      7132816      7132816
2     ad_spend  11108749.09  11108749.09
3  conversions       326812       326812
4      revenue  54183330.81  54183330.81
                          check  passed
0           row_count_preserved    True
1         observation_id_unique    True
2                columns_unique    True
3              source_unchanged    True
4   impressions_total_preserved    True
5        clicks_total_preserved    True
6      ad_spend_total_preserved    True
7   conversions_total_preserved    True
8       revenue_total_preserved    True
9          CTR_source_preserved    True
10         CPC_source_preserved    True
11         CPA_source_preserved    True
12        ROAS_source_preserved    True
                              flag  flagged  not_evaluated
0               flag_ctr_undefined        0              0
1         flag_ctr_source_mismatch       47              0
2          flag_c

## 7. Load — xuất duy nhất một CSV và kiểm tra đọc lại

Xuất UTF-8 theo cấu hình. Ngày ghi theo định dạng nguồn đã khai báo; số tiền và tỷ lệ được biểu diễn từ Decimal, không qua float. Cờ chất lượng là True/False; trạng thái chưa đối chiếu được để trống.

Ghi vào file tạm, đọc lại để kiểm tra số dòng/cột, mã quan sát và các tổng. Chỉ sau khi kiểm tra đạt mới thay thế file đầu ra chính thức. File tạm luôn được dọn; chạy lại cùng nguồn/cấu hình ghi vào cùng đường dẫn, không sinh bản theo timestamp.

CSV không lưu schema kiểu dữ liệu hoặc quan hệ Star Schema. Khi nhập vào BI, khai báo kiểu và dùng năm đại lượng cơ sở để tính KPI theo bộ lọc. Đây là ETL bảng chi tiết để kiểm tra trước, chưa phải triển khai PostgreSQL hoặc mô phỏng DSS.

In [9]:
output_dir.mkdir(parents=True, exist_ok=True)
to_export = clean.copy()
to_export[date_col] = to_export[date_col].dt.strftime(cfg['source']['date_format'])
temp_path = None
try:
    with tempfile.NamedTemporaryFile(mode='w', encoding=cfg['export']['encoding'], newline='',
                                     dir=output_dir, suffix='.tmp', delete=False) as file:
        temp_path = Path(file.name)
        to_export.to_csv(file, index=False)
    reloaded = pd.read_csv(temp_path, dtype=str, keep_default_na=False, encoding=cfg['export']['encoding'])
    if len(reloaded) != len(raw) or list(reloaded.columns) != list(clean.columns):
        raise AssertionError('CSV đọc lại sai số dòng hoặc schema.')
    if reloaded[derived['observation_id']].tolist() != clean[derived['observation_id']].tolist():
        raise AssertionError('CSV không giữ mã/thứ tự quan sát.')
    for role in cfg['measures']:
        if sum((Decimal(v) for v in reloaded[col[role]]), Decimal(0)) != sum(
                (Decimal(str(v)) for v in clean[col[role]]), Decimal(0)):
            raise AssertionError(f'Tổng {role} sai sau khi đọc lại CSV.')
    os.replace(temp_path, output_path)
finally:
    if temp_path is not None and temp_path.exists():
        temp_path.unlink()
display(Markdown(f'Đã xuất **{len(clean):,} quan sát**, **{len(clean.columns)} cột** vào '
                 f'`{output_path.relative_to(ROOT)}`. **{len(checks)} kiểm tra trước xuất đạt**; '
                 'đã đọc lại và đối chiếu các tổng. Không xuất thêm bảng trung gian.'))

<IPython.core.display.Markdown object>
